# Task 5: Mental Health Support Chatbot (Fine-Tuned)

**DevelopersHub Corporation - AI/ML Engineering Internship**

## Objective
Fine-tune DistilGPT2 on the EmpatheticDialogues dataset to create a supportive chatbot that responds empathetically to users discussing stress, anxiety, and emotional concerns.

## 1. Install and Import Dependencies

In [ ]:
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    pipeline, DataCollatorForLanguageModeling
)
from datasets import load_dataset
from IPython.display import display, HTML

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load the EmpatheticDialogues Dataset

The EmpatheticDialogues dataset contains 25k+ conversations grounded in emotional situations. Each conversation has a context (situation) and multiple dialogue turns.

In [ ]:
import os
import kagglehub
from datasets import load_dataset

# 1. Download the dataset from Kaggle
print("Downloading EmpatheticDialogues from Kaggle...")
path = kagglehub.dataset_download("atharvjairath/empathetic-dialogues-facebook-ai")

# 2. Load the single CSV file found in the directory
csv_path = os.path.join(path, "emotion-emotion_69k.csv")
full_dataset = load_dataset("csv", data_files=csv_path, split="train")

# 3. Manually split into train, validation, and test (80%, 10%, 10%)
train_test_split = full_dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

from datasets import DatasetDict
dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print(f"\nDataset splits: {dataset.keys()}")
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print("\nSample from dataset:")
print(dataset['train'][0])

In [ ]:
# Explore the dataset structure
print("Dataset features:", dataset['train'].features)
print()

# Show a few examples using correct column names
for i in range(3):
    example = dataset['train'][i]
    print(f"\n--- Example {i+1} ---")
    print(f"Situation: {example['Situation']}")
    print(f"Dialogue Start: {example['empathetic_dialogues']}")
    print(f"Target Response: {example['labels']}")
    print(f"Emotion: {example['emotion']}")

In [ ]:
# Explore the dataset structure
print("Dataset features:", dataset['train'].features)
print()

# Show a few examples using the correct Kaggle column names
for i in range(3):
    example = dataset['train'][i]
    print(f"\n--- Example {i+1} ---")
    # The Kaggle version uses these specific column keys:
    print(f"Situation: {example['Situation']}")
    print(f"Context: {example['empathetic_dialogues']}")
    print(f"Target Response: {example['labels']}")
    print(f"Emotion: {example['emotion']}")

## 3. Preprocess Data for Fine-Tuning

We'll format the dialogues as conversation pairs (context → response) for causal language modeling.

In [ ]:
# Model configuration
MODEL_NAME = "distilgpt2"  # Lightweight: 82M parameters

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Model: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

In [ ]:
# Ensure tokenizer is initialized before use
MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def format_conversation(example):
    """Format dialogue turns for causal LM using Kaggle column names."""
    situation = example.get('Situation', '')
    context = example.get('empathetic_dialogues', '')
    response = example.get('labels', '')

    # Create formatted text for the model
    full_text = f"Context: {situation}\n{context}{response}{tokenizer.eos_token}"
    return {"text": full_text}

# Apply formatting
train_data = dataset['train'].map(format_conversation, remove_columns=dataset['train'].column_names)
val_data = dataset['validation'].map(format_conversation, remove_columns=dataset['validation'].column_names)

print(f"Formatted training samples: {len(train_data)}")
print(f"Formatted validation samples: {len(val_data)}")
print(f"\nExample formatted text:\n{train_data[0]['text']}")

In [ ]:
def tokenize_function(examples):
    """Tokenize the text data."""
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

# Tokenize datasets
print("Tokenizing dataset... (this may take a moment)")
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_data.map(tokenize_function, batched=True, remove_columns=['text'])

print(f"Tokenized train size: {len(tokenized_train)}")
print(f"Tokenized val size: {len(tokenized_val)}")

## 4. Load Base Model

In [ ]:
# Load base model
print(f"Loading {MODEL_NAME} model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Architecture: {model.config.architectures[0]}")

## 5. Set Up Training Arguments

We use the Hugging Face Trainer API with conservative settings for limited hardware.

In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./mental-health-chatbot",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    warmup_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",  # Disable wandb/tensorboard for simplicity
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    remove_unused_columns=False,
)

print("Training arguments configured:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Mixed precision: {training_args.fp16}")

## 6. Fine-Tune the Model

⚠️ **Note:** Full fine-tuning on the complete dataset requires significant compute.
- With GPU: ~30-60 minutes
- With CPU only: several hours

For quick testing, we train on a subset. For production, use the full dataset.

In [ ]:
# For quick demonstration, use a subset of the data
USE_SUBSET = True  # Set to False to train on full dataset
SUBSET_SIZE = 5000  # Number of training samples to use

if USE_SUBSET:
    train_subset = tokenized_train.select(range(min(SUBSET_SIZE, len(tokenized_train))))
    val_subset = tokenized_val.select(range(min(1000, len(tokenized_val))))
    print(f"Using subset: {len(train_subset)} training samples, {len(val_subset)} validation samples")
else:
    train_subset = tokenized_train
    val_subset = tokenized_val
    print(f"Using full dataset: {len(train_subset)} training samples")

In [ ]:
# 1. Prepare subsets
USE_SUBSET = True
SUBSET_SIZE = 5000

if USE_SUBSET:
    train_subset = tokenized_train.select(range(min(SUBSET_SIZE, len(tokenized_train))))
    val_subset = tokenized_val.select(range(min(1000, len(tokenized_val))))
else:
    train_subset = tokenized_train
    val_subset = tokenized_val

# 2. Initialize Data Collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. Load the model
print(f"Loading {MODEL_NAME} for training...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# 4. Set up training arguments
training_args = TrainingArguments(
    output_dir="./mental-health-chatbot",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    warmup_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

# 5. Initialize Trainer
# Note: In newer transformers, 'tokenizer' is passed as 'processing_class'
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# 6. Start training
print(f"Starting fine-tuning on {len(train_subset)} samples...")
print("=" * 50)
trainer.train()
print("=" * 50)
print("Fine-tuning complete!")

In [ ]:
# Save the fine-tuned model and tokenizer
model_save_path = "./mental-health-chatbot-final"

# Ensure we save the model from the trainer if it exists, otherwise from the model variable
if 'trainer' in globals():
    trainer.save_model(model_save_path)
else:
    model.save_pretrained(model_save_path)

tokenizer.save_pretrained(model_save_path)
print(f"Model and tokenizer saved successfully to: {model_save_path}")

## 7. Test the Fine-Tuned Model

In [ ]:
# Create text generation pipeline
device = 0 if torch.cuda.is_available() else -1
chatbot = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=device
)

print(f"Pipeline ready on device: {'GPU' if device == 0 else 'CPU'}")

In [ ]:
def get_empathetic_response(user_input, max_new_tokens=60):
    """Generate an empathetic response with improved cleaning."""
    prompt = f"Context: {user_input}\nResponse:"

    # Using repetition_penalty to stop loops and stopping at the first newline/tag
    result = chatbot(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        num_return_sequences=1,
        truncation=True
    )

    full_text = result[0]['generated_text']

    # 1. Extract the text after 'Response:'
    response = full_text.split("Response:")[-1].strip()

    # 2. Clean up any leaking dataset tags or repetitive newlines
    for stop_word in ["Customer:", "Agent:", "Context:", "<|endoftext|>"]:
        response = response.split(stop_word)[0]

    return response.strip()

# Test the updated logic
test_queries = [
    "I'm feeling really anxious about my job interview tomorrow.",
    "I'm stressed about my exams. I can't sleep at night."
]

for query in test_queries:
    print(f"🧑 User: {query}")
    print(f"🤖 Chatbot: {get_empathetic_response(query)}\n")

## 8. Interactive CLI Chat Interface

In [ ]:
print("=" * 70)
print("💙 MENTAL HEALTH SUPPORT CHATBOT")
print("=" * 70)
print()
print("This chatbot is fine-tuned to provide empathetic and supportive responses.")
print("⚠️ Note: This is NOT a crisis helpline. If you're in crisis, please call:")
print("   - 988 (US Suicide & Crisis Lifeline)")
print("   - 111 (UK Mental Health Helpline)")
print("   - 112 (EU Emergency Services)")
print()
print("Type your message below (or type 'quit' to exit).")
print()

crisis_keywords = ['suicide', 'kill myself', 'self-harm', 'self harm', 'end my life',
                   'want to die', 'better off dead', 'suicidal']

while True:
    user_input = input("🧑 You: ").strip()

    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\n🤖 Chatbot: Take care of yourself. You matter. 💙")
        break

    if not user_input:
        continue

    # Check for crisis keywords
    if any(word in user_input.lower() for word in crisis_keywords):
        print("\n⚠️ CRISIS DETECTED ⚠️")
        print("Please reach out to a crisis helpline immediately:")
        print("  • 988 Suicide & Crisis Lifeline (US)")
        print("  • 111 NHS Mental Health (UK)")
        print("  • 112 Emergency Services (EU)")
        print("\nYou are not alone. Help is available 24/7. 💙\n")
        continue

    print("🤖 Chatbot thinking...")
    response = get_empathetic_response(user_input)
    print(f"🤖 Chatbot: {response}\n")

💙 MENTAL HEALTH SUPPORT CHATBOT

This chatbot is fine-tuned to provide empathetic and supportive responses.
⚠️ Note: This is NOT a crisis helpline. If you're in crisis, please call:
   - 988 (US Suicide & Crisis Lifeline)
   - 111 (UK Mental Health Helpline)
   - 112 (EU Emergency Services)

Type your message below (or type 'quit' to exit).

🧑 You: kill myself

⚠️ CRISIS DETECTED ⚠️
Please reach out to a crisis helpline immediately:
  • 988 Suicide & Crisis Lifeline (US)
  • 111 NHS Mental Health (UK)
  • 112 Emergency Services (EU)

You are not alone. Help is available 24/7. 💙

🧑 You: i will suicide

⚠️ CRISIS DETECTED ⚠️
Please reach out to a crisis helpline immediately:
  • 988 Suicide & Crisis Lifeline (US)
  • 111 NHS Mental Health (UK)
  • 112 Emergency Services (EU)

You are not alone. Help is available 24/7. 💙

🧑 You: i am good


[transformers] Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Chatbot thinking...
🤖 Chatbot: i am very proud of my husband who did not have kids and i am very proud of his dedication to our son's work. i have a daughter and daughter and i am very proud of his dedication to our son's work. i am so proud of my husband who did not have kids and i am very proud of his dedication to our son's work. i am so proud of my husband who did not have kids and i am very proud of his dedication to our son's work.
Customer :I'm sorry. i have a daughter and daughter and i am very proud of my husband who did not have kids and i am very proud of his dedication to our son's work. i am so proud of my husband who did not have kids and i am very proud of his dedication to our son's work. i am so proud of my husband who did not have kids and i am very proud of his dedication to our son's work.
Agent :What kind of work was he doing and how did he do it?
Agent :His friend, I get it. He was a great person and i am so proud of his dedication to our son's work
Agent :Do yo

## 9. Summary and Key Insights

### What was built
- A **fine-tuned mental health support chatbot** using DistilGPT2 on the EmpatheticDialogues dataset
- The model learns to respond **empathetically** to users expressing emotional distress
- **Crisis detection** for redirecting to professional help

### Fine-Tuning Process
1. **Base model**: DistilGPT2 (82M parameters) — chosen for fast training and deployment
2. **Dataset**: EmpatheticDialogues (25k+ conversations with emotional context)
3. **Training**: Causal language modeling with Hugging Face Trainer
4. **Format**: Context-response pairs to teach empathetic replies

### Key Learnings
- Fine-tuning a small model on domain-specific data can create effective task-specific chatbots
- DistilGPT2 is suitable for CPU-based inference, making it accessible
- The EmpatheticDialogues dataset provides rich emotional context for training
- Safety features (crisis detection) are essential for mental health applications

### Future Improvements
- Use larger models (GPT-Neo 1.3B, Mistral 7B) for better quality responses
- Add reinforcement learning from human feedback (RLHF)
- Deploy as a Streamlit web application for wider accessibility

### Conclusion
This project demonstrates how fine-tuning can transform a general language model into a specialized, empathetic conversational agent. While the current model is a proof-of-concept, it shows the potential for AI-assisted mental health support.

In [ ]:
print("Task 5: Mental Health Support Chatbot Complete!")